In [1]:
import yaml
import re
import pandas as pd

from pathlib import Path
from textwrap import fill
from tqdm import tqdm
from utils import text_cleaning, normalize_sentence, classify_sentence

%load_ext autoreload
%autoreload 2

In [2]:
TOKEN_EXTRACTION_CONFIG_PATH = Path.cwd().parent / 'configs' / 'token_extraction.yaml'
with TOKEN_EXTRACTION_CONFIG_PATH.open(encoding='utf-8') as f:
    extraction_config = yaml.safe_load(f)

REQUIREMENTS_KEY_WORDS = extraction_config.get("REQUIREMENTS_KEY_WORDS", [])
RESPONSIBILITIES_KEY_WORDS = extraction_config.get("RESPONSIBILITIES_KEY_WORDS", [])
CONDITIONS_KEY_WORDS = extraction_config.get("CONDITIONS_KEY_WORDS", [])

NECESSARY_PUNCTUATION = extraction_config.get("NECESSARY_PUNCTUATION", [])
PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES = extraction_config.get("PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES", [])

PLUSES = extraction_config.get("PLUSES", [])
MINUSES = extraction_config.get("MINUSES", [])
DOTS = extraction_config.get("DOTS", [])

RUBBISH_TAGS = extraction_config.get("RUBBISH_TAGS", [])
TAGS_TO_SAVE = extraction_config.get("TAGS_TO_SAVE", [])

In [3]:
cleaned_vacancies = pd.read_csv("../headhunter_parsing/data/vacancies_cleaned.csv")

### Вычислим множество ненужных символов пунктуации

In [4]:
punctuation = set()
for text in cleaned_vacancies.description:
    new_symbols = set(re.findall(r'[^\wё]', text.lower()))
    punctuation |= new_symbols

RUBBISH_PUNCTUATION = punctuation - set(NECESSARY_PUNCTUATION)
print(fill(", ".join(RUBBISH_PUNCTUATION), 70))

🎉, ⚓, ÷, 🗃, 🎓, 🪷, , ”, 📩, 🕘, ☕, ▎, ↴, 🐳, ☘, 🎁, 🌏, ≠, ‌, ⏳, ▪, 🍎, 🍃,
😎, “, 🫀, ⛱, 😀, 🌎, 🛡, ☑, 🌭, 🏻, , ❌, ✂, 🪟, 💎, ⛽, ✨, ▶, 👶, ~, ☀, 🎛, 🌙,
🏢, 📥, 🤞, 🌸, 💐, 🫶, 　, ⁤, ⟶, ⸻, 🔍, ✅, 💚, ¬, 🟢, 🇬, ₾, 🔹, 🐍, 👨, 👬, ¸, 🏆,
®, ◾, 💵, 🍕, 🔵, 🦾, 🌍, 🏭, 💳, ♦, •, ☝, ◦, …, ✌, 🧾, 🩷, ], 📝, 📸, 🕗, ♂, ̈,
`, 🧐, ❖, 📋, 🏫, 📍, ⭐, ⚖, 👍, 📅, 📡, 🔎, 🇺, –, 🔒, ,, 👥, €, 🏙, 👇, 🕰, 💊, ♀,
👩, °, 🔶, ⏬, 🦷, 😄, 🙂, 🪡, », 🔝, „, ⚡, ), ❣, 🕐, 🖨, ◽, 🖤, 🌼, 😌, 🥳, ✿, 🕒,
⎯, 🤝, 🎈, 😊, 🥰, 🏖, 🧰, ↓, 🗓, 🥐, 😏, ﻿, ›, ✈, 💬, ‍, ’, 👌, 🏄, ️, 💯, 🧘, 🧙,
👋, ↔, 🏠, 🛋, ⌨, 📚, 💙, ‎, ❎, 📲, 🌊, 🚌, 🏋, 🙌, 🧹, ➔, 📣, ⛏, 🇪, ⚔, 💻,  , ,
©, 🔌, 🚫, ◎, 🐶, 📧, 🕓, 🤗, ͟, 🌿, ■, 😉, ̇, 🎂, 🍖, , 🇷, 🔆, ⚜, ^, }, 👤, 🙏,
👀, |, 👉, ∙, 🚚, 🤖, ‑, ➠, 🎥, 📑, *, №, 🔑, ⦁, 🌹, ✉, 🏦, 🇰, −, 【, [, ̆, 👦,
🏅, 🕑, 🌴, 📞, ★, 💰, 🧑, 💗, ：, 🎯,  , ⁄, 🚛, ₸, 🪨, \, →, , ·, 🏼, 💜, 🎬, ⃣,
￼, ⬇, ━, 🖥, 📂, 📦, , 】, 🎄, ❤, 🏊,  , ",  , 🏽, ☆, 🔘,  , 💥, !, 🤍, ❄, 🏥,
🌞, ❕, ►, 🐧, 🚨, 🧁, ✍, 🍽, 💧, =, ≪, ⁢, ☺, 🇿, 🥈, 🧪, ─, ₽, —, ⁣, ，, ♾, 💕,
😍, ♻, ­, 🛠, ⏺, ―, ←, ⚙, ✏, ​, ✳, 📈, 🔬, 🕙, 🍬, ○, 🚕, 🤔, 🖌, ?, 🎨, 🧩, ✔,
📁, 🥗, ☠, ±, 📢, ‒, ́, 📎, 📘, 📐, ➢, 🤸

### Извлечём токены Требований, Обязанностей и Условий из текстов вакансий

In [8]:
rows = []
columns = list(cleaned_vacancies.columns)

missing_req, missing_res, missing_con = 0, 0, 0
omissions = 0
good_descriptions = 0

for tuple in tqdm(cleaned_vacancies.itertuples(index = False, name = None)):
    row = dict(zip(columns, tuple))
    id = row.get("id", "")
    description = row.get("description", "")

    req, res, con = list(), list(), list()

    cleaned_description = text_cleaning(
        description, PLUSES, DOTS, MINUSES, RUBBISH_TAGS, RUBBISH_PUNCTUATION, 
        TAGS_TO_SAVE, PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES
        )

    sentences = re.findall(r"(?:<[\w\s/]+>)?([\w\s\-:./%+$#]+)(?:<[\w\s/]+>)?", cleaned_description)
    sentence_types = [0, 0, 0]

    for sentence in sentences:
        if not sentence.strip():
            continue
        normalized_sentence = normalize_sentence(sentence)
        is_change, new_types = classify_sentence(
            normalized_sentence, REQUIREMENTS_KEY_WORDS,
            RESPONSIBILITIES_KEY_WORDS, CONDITIONS_KEY_WORDS
            )
        if is_change:
            sentence_types = new_types
        if sentence_types[0] == 1:
            req.append(normalized_sentence)
        if sentence_types[1] == 1:
            res.append(normalized_sentence)
        if sentence_types[2] == 1:
            con.append(normalized_sentence)
        
    req = list(req)
    res = list(res)
    con = list(con)

    if len(req) * len(res) * len(con) == 0:
        missing_req += (len(req) == 0)
        missing_res += (len(res) == 0)
        missing_con += (len(con) == 0)
        omissions += 1
        continue
    else:
        good_descriptions += 1
        rows.append([id, req, res, con])

print(f'''
Descriptions with all 3 aspect groups presented: {good_descriptions} ✔️
Descriptions with at least 1 aspect group missing: {omissions} ❌
Descriptions with missing requirements: {missing_req}
Descriptions with missing responsibilities: {missing_res}
Descriptions with missing conditions: {missing_con}

Share of successfully processed descriptions: {round(good_descriptions / len(cleaned_vacancies) * 100, 2)}%
''')

resulting_table = pd.DataFrame(rows, columns=['id', 'requirements', 'responsibilities', 'conditions'])
resulting_table.to_csv('data/rrs.csv', index = False)

0it [00:00, ?it/s]

36374it [01:37, 374.04it/s]



Descriptions with all 3 aspect groups presented: 33612 ✔️
Descriptions with at least 1 aspect group missing: 2762 ❌
Descriptions with missing requirements: 637
Descriptions with missing responsibilities: 1142
Descriptions with missing conditions: 2125

Share of successfully processed descriptions: 92.41%

